In [1]:
import tensorflow as tf
import numpy as np
import os

print("TensorFlow version:", tf.__version__)

print("\nGPU devices:")
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("✅ GPU available!")
    for gpu in gpus:
        print(gpu)
else:
    print("⚠️ GPU not detected. Training will use CPU.")

print("\nNumPy version:", np.__version__)

TensorFlow version: 2.20.0

GPU devices:
✅ GPU available!
PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

NumPy version: 2.0.2


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

BASE_PATH = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"

for root, dirs, files in os.walk(BASE_PATH):
    for file in files:
        if file == "processed_potato_data.npz":
            print("✅ FILE FOUND:")
            print(os.path.join(root, file))

✅ FILE FOUND:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing/processed_potato_data.npz


In [4]:
import os

BASE_PATH = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"

for root, dirs, files in os.walk(BASE_PATH):
    for file in files:
        if file == "processed_potato_data.npz":
            print("✅ FILE FOUND:")
            print(os.path.join(root, file))

✅ FILE FOUND:
/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing/processed_potato_data.npz


In [5]:
# ==========================================
# CELL 5: DATA LOADING & CLASS WEIGHT CALCULATION
# ==========================================
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

PROCESSED_PATH = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing/processed_potato_data.npz"

data = np.load(PROCESSED_PATH)

X_train = data["X_train"]
y_train = data["y_train"]

X_val = data["X_val"]
y_val = data["y_val"]

X_test = data["X_test"]
y_test = data["y_test"]

print("✅ Preprocessed dataset loaded successfully!\n")
print("Train      :", X_train.shape, y_train.shape)
print("Validation :", X_val.shape, y_val.shape)
print("Test       :", X_test.shape, y_test.shape)

class_names = ["Healthy", "Early_Blight", "Late_Blight"]

# Calculate class weights to handle Healthy class imbalance
classes = np.unique(y_train)
class_weights_arr = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, class_weights_arr))

print("\n========== CLASS WEIGHTS (HANDLING IMBALANCE) ==========")
for cls_idx, weight in class_weights.items():
    print(f"Class {cls_idx} ({class_names[cls_idx]:12s}): Weight = {weight:.4f}")
print("\nClass Weights Dict:", class_weights)

✅ Preprocessed dataset loaded successfully!

Train      : (1721, 224, 224, 3) (1721,)
Validation : (215, 224, 224, 3) (215,)
Test       : (216, 224, 224, 3) (216,)

========== CLASS WEIGHTS (HANDLING IMBALANCE) ==========
Class 0 (Healthy     ): Weight = 4.7410
Class 1 (Early_Blight): Weight = 0.7171
Class 2 (Late_Blight ): Weight = 0.7171

Class Weights Dict: {np.int64(0): np.float64(4.741046831955923), np.int64(1): np.float64(0.7170833333333333), np.int64(2): np.float64(0.7170833333333333)}


In [6]:
# ==========================================
# CELL 6: BUILD & COMPILE CNN ARCHITECTURE (Potato_Leaf_CNN)
# ==========================================
import tensorflow as tf
from tensorflow.keras import layers, models

def build_potato_leaf_cnn(input_shape=(224, 224, 3), num_classes=3):
    model = models.Sequential([
        # Input Block
        layers.Input(shape=input_shape),

        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Block 4
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Fully Connected Header
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ], name="Potato_Leaf_CNN")

    return model

# Instantiate & Compile
model = build_potato_leaf_cnn(input_shape=(224, 224, 3), num_classes=3)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("========== Potato_Leaf_CNN SUMMARY ==========")
model.summary()

========== Potato_Leaf_CNN SUMMARY ==========


Model: "Potato_Leaf_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,665,795 (25.43 MB)

 Trainable params: 6,665,091 (25.43 MB)

 Non-trainable params: 704 (2.75 KB)

In [7]:
# ==========================================
# CELL 7: CHECKPOINT SETUP & MODEL TRAINING
# ==========================================
import os
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# Save Directory & Model Path
SAVE_DIR = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model"
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, "potato_cnn_best.keras")

print("✅ Checkpoint directory:", SAVE_DIR)
print("✅ Target model path    :", MODEL_SAVE_PATH)

# Callbacks
checkpoint_cb = ModelCheckpoint(
    filepath=MODEL_SAVE_PATH,
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stopping_cb = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_cb = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

# Start Training
EPOCHS = 25
BATCH_SIZE = 32

print("\n🚀 Starting Potato_Leaf_CNN Training...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=[checkpoint_cb, early_stopping_cb, reduce_lr_cb],
    verbose=1
)

print("\n🎉 Training Complete! Best model saved to:", MODEL_SAVE_PATH)

✅ Checkpoint directory: /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model
✅ Target model path    : /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/potato_cnn_best.keras

🚀 Starting Potato_Leaf_CNN Training...
Epoch 1/25
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step - accuracy: 0.7772 - loss: 3.9253
Epoch 1: val_accuracy improved from None to 0.46512, saving model to /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/potato_cnn_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/potato_cnn_best.keras
54/54 ━━━━━━━━━━━━━━━━━━━━ 33s 364ms/step - accuracy: 0.8559 - loss: 3.2509 - val_accuracy: 0.4651 - val_loss: 66.0689 - learning_rate: 0.0010
Epoch 2/25
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.9034 - loss: 2.5210
Epoch 2: val_accuracy did not improve from 0.46512
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 76ms/step - ac

In [8]:
# ==========================================
# STEP 1: RELOAD SAVED MODEL & VERIFY
# ==========================================

import tensorflow as tf
import numpy as np
import os

MODEL_PATH = "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/potato_cnn_best.keras"

print("Checking saved model...")

# 1. Check file exists
print("\nFile exists:", os.path.exists(MODEL_PATH))

if os.path.exists(MODEL_PATH):
    print("File size:", round(os.path.getsize(MODEL_PATH) / (1024 * 1024), 2), "MB")

# 2. Load model from saved file
loaded_model = tf.keras.models.load_model(MODEL_PATH)

print("\n✅ Model loaded successfully!")

# 3. Display model information
print("\nModel name:", loaded_model.name)
print("Input shape:", loaded_model.input_shape)
print("Output shape:", loaded_model.output_shape)
print("Total parameters:", loaded_model.count_params())

Checking saved model...

File exists: True
File size: 76.35 MB

✅ Model loaded successfully!

Model name: Potato_Leaf_CNN
Input shape: (None, 224, 224, 3)
Output shape: (None, 3)
Total parameters: 6665795


In [9]:
# ==========================================
# STEP 2: PREDICTION USING RELOADED MODEL
# ==========================================

class_names = ["Healthy", "Early_Blight", "Late_Blight"]

# Take one test image
sample_image = X_test[0]
actual_label = y_test[0]

# Add batch dimension
sample_batch = np.expand_dims(sample_image, axis=0)

# Prediction
prediction = loaded_model.predict(sample_batch, verbose=0)

predicted_class = np.argmax(prediction[0])
confidence = np.max(prediction[0]) * 100

print("========== MODEL PREDICTION ==========")
print("Actual Class    :", class_names[actual_label])
print("Predicted Class :", class_names[predicted_class])
print("Confidence      :", f"{confidence:.2f}%")
print("Probabilities   :", prediction[0])

if predicted_class == actual_label:
    print("\n✅ Prediction is CORRECT")
else:
    print("\n⚠️ Prediction is INCORRECT")

========== MODEL PREDICTION ==========
Actual Class    : Late_Blight
Predicted Class : Late_Blight
Confidence      : 100.00%
Probabilities   : [5.4663267e-18 5.1290070e-12 1.0000000e+00]

✅ Prediction is CORRECT


In [10]:
# ==========================================
# STEP 3: TEST SET PREDICTION
# ==========================================

test_loss, test_accuracy = loaded_model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("\n========== FINAL TEST RESULT ==========")
print("Test Loss     :", test_loss)
print("Test Accuracy :", f"{test_accuracy * 100:.2f}%")

7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 269ms/step - accuracy: 0.9815 - loss: 0.0952

========== FINAL TEST RESULT ==========
Test Loss     : 0.0951717421412468
Test Accuracy : 98.15%
